# Etapa 1 — Indução simbólica, ganho de informação e monotonicidade

A classificação principal é `alto=1` quando a contagem UF–evento–mês excede o percentil 75 de seu indicador, estimado **apenas no treino**. A análise é agregada; o tamanho da UF influencia fortemente a classe.

**Fonte:** BancoVDE 2026.xlsx / MJSP. **Semente:** 42.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_root():
    """Localiza a raiz do projeto (pasta com scripts/utils.py e data/) a partir do diretório atual."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "scripts" / "utils.py").exists() and (base / "data").exists():
            return base
    for pattern in ("*/scripts/utils.py", "*/*/scripts/utils.py"):   # ex.: Colab em /content
        for hit in sorted(here.glob(pattern)):
            if (hit.parent.parent / "data").exists():
                return hit.parent.parent
    raise FileNotFoundError(
        "Raiz do projeto não encontrada: abra o notebook com o diretório atual "
        "dentro da pasta FIA_P3_SegurancaPublica_Equipe5."
    )

ROOT = _find_root()

sys.path.insert(0, str(ROOT / "scripts"))

sys.modules.pop("utils", None)

from utils import (
    panel,
    training_split,
    prepare_splits,
    temporal_split,
    write_metrics
)

np.random.seed(42)
(ROOT / "outputs").mkdir(exist_ok=True)

print("Configuração da Etapa 1 concluída!")

Configuração da Etapa 1 concluída!


In [2]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from utils import ID3, ent, gain, class_features, CATEGORICAS
p,tr,te,meta=training_split()
Xtr=class_features(tr);Xte=class_features(te)
ytr=tr.alto;yte=te.alto
print('Painel:',len(p),'treino:',len(tr),'teste:',len(te),'classes:',ytr.value_counts(normalize=True).to_dict())
print('Limiar por indicador estimado no treino:',meta['p75_train_por_evento'])

Painel: 2590 treino: 2072 teste: 518 classes: {0: 0.7755791505791506, 1: 0.22442084942084942}
Limiar por indicador estimado no treino: {'Estupro': 83.5, 'Estupro de vulnerável': 286.0, 'Feminicídio': 7.0, 'Furto de veículo': 475.0, 'Homicídio doloso': 128.0, 'Lesão corporal seguida de morte': 2.0, 'Roubo a instituição financeira': 0.0, 'Roubo de carga': 5.5, 'Roubo de veículo': 192.0, 'Roubo seguido de morte (latrocínio)': 3.0, 'Tentativa de feminicídio': 18.0, 'Tentativa de homicídio': 130.0}


## Entropia e os três primeiros níveis
Calculamos diretamente $H(Y)=-\sum_{y} p(y)\log_2 p(y)$ e $G(Y,A)=H(Y)-\sum_{a}p(a)H(Y\mid A=a)$. A tabela contém todas as alternativas avaliadas nos nós dos níveis 1–3, não só a raiz. O relatório apresenta uma trilha completa de três níveis.

In [3]:
records=[]
def trace(data, attrs, depth=1, path='raiz'):
 if depth>3 or data.alto.nunique()<2 or not attrs:return
 root_h=ent(data.alto)
 stats={f:gain(data,f,'alto') for f in attrs}
 best=max(stats,key=stats.get)
 for f,g in stats.items():
  records.append({'nivel':depth,'caminho':path,'n':len(data),'n1':int(data.alto.sum()),
  'h_pai':root_h,'atributo':f,'h_filhos_ponderada':root_h-g,'ganho':g,'escolhido':f==best})
 for value,part in data.groupby(best):
  trace(part.drop(columns=[best]),[f for f in attrs if f!=best],depth+1,path+f'/{best}={value}')
a=Xtr.copy();a['alto']=ytr.to_numpy()
trace(a,list(Xtr));table=pd.DataFrame(records)
table.to_csv(ROOT/'outputs'/'ganhos_niveis_1_a_3.csv',index=False)
print(table[table.escolhido].head(25).round(5).to_string(index=False))
assert set(table.nivel)=={1,2,3}

 nivel                                               caminho    n  n1   h_pai    atributo  h_filhos_ponderada   ganho  escolhido
     1                                                  raiz 2072 465 0.76816          uf             0.40234 0.36582       True
     2                                            raiz/uf=AM   72   3 0.24988      evento             0.10605 0.14383       True
     3 raiz/uf=AM/evento=Roubo seguido de morte (latrocínio)    8   3 0.95443 alta_previa             0.60684 0.34759       True
     2                                            raiz/uf=BA   69  61 0.51758      evento             0.24871 0.26887       True
     3                             raiz/uf=BA/evento=Estupro    7   6 0.59167   semestre2             0.39356 0.19812       True
     3                         raiz/uf=BA/evento=Feminicídio    6   4 0.91830   semestre2             0.66667 0.25163       True
     3 raiz/uf=BA/evento=Roubo seguido de morte (latrocínio)    5   4 0.72193 alta_previa        

## Árvore do zero, biblioteca e cinco dobras sem contaminação
Em cada dobra o P75 e a mediana do atributo lag são **recalculados exclusivamente com a parte de treino**. São exibidas acurácias no treino, no teste e baselines global e de maioria por UF.

In [4]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from utils import ID3, ent, gain, class_features, CATEGORICAS

id3 = ID3(max_depth=4)
id3.fit(Xtr, ytr)

base = make_pipeline(
    OneHotEncoder(handle_unknown='ignore'),
    DecisionTreeClassifier(
        max_depth=4,
        criterion='entropy',
        random_state=42
    )
)
base.fit(Xtr, ytr)

tr_acc = accuracy_score(ytr, id3.predict(Xtr))
id3_acc = accuracy_score(yte, id3.predict(Xte))
sk_acc = base.score(Xte, yte)

mode = int(ytr.mean() >= 0.5)
bacc = accuracy_score(yte, np.full(len(te), mode))

ufmajor = tr.groupby('uf').alto.mean().ge(0.5).astype(int)
ufpred = te.uf.map(ufmajor).fillna(mode).astype(int)
ufacc = accuracy_score(yte, ufpred)

scores = []
fold_baselines = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (itr, ivl) in enumerate(kf.split(p), 1):
    tt, vv, _ = prepare_splits(p, itr, ivl)

    foldmodel = make_pipeline(
        OneHotEncoder(handle_unknown='ignore'),
        DecisionTreeClassifier(
            max_depth=4,
            criterion='entropy',
            random_state=42
        )
    )

    foldmodel.fit(class_features(tt), tt.alto)

    scores.append(
        foldmodel.score(class_features(vv), vv.alto)
    )

    majority = int(tt.alto.mean() >= 0.5)

    fold_baselines.append(
        accuracy_score(vv.alto, np.full(len(vv), majority))
    )

print("ID3 treino:", round(tr_acc, 4))
print("ID3 teste:", round(id3_acc, 4))
print("Sklearn teste:", round(sk_acc, 4))

print("\n5-fold real:", np.round(scores, 4))
print("Média:", np.mean(scores))
print("Baseline CV:", np.mean(fold_baselines))

print("\nBaseline teste:", bacc)
print("Baseline UF:", ufacc)

pred = base.predict(Xte)

print("\nMatriz de confusão sklearn:")
print(confusion_matrix(yte, pred))

print("\nRelatório de classificação:")
print(classification_report(yte, pred, zero_division=0))

stats = (
    tr.groupby('uf')
    .alto.agg(['count', 'mean'])
    .sort_values('mean', ascending=False)
)

print("\nDistribuição por UF (treino):")
print(stats.to_string())

ID3 treino: 0.9392
ID3 teste: 0.9344
Sklearn teste: 0.8938

5-fold real: [0.8938 0.8591 0.8417 0.8958 0.8282]
Média: 0.8637065637065637
Baseline CV: 0.7725868725868726

Baseline teste: 0.7972972972972973
Baseline UF: 0.8803088803088803

Matriz de confusão sklearn:
[[409   4]
 [ 51  54]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94       413
           1       0.93      0.51      0.66       105

    accuracy                           0.89       518
   macro avg       0.91      0.75      0.80       518
weighted avg       0.90      0.89      0.88       518


Distribuição por UF (treino):
    count      mean
uf                 
SP     79  0.911392
RJ     73  0.904110
BA     69  0.884058
MG     82  0.658537
PE     85  0.494118
PR     75  0.440000
PA     77  0.363636
MA     79  0.303797
RS     76  0.302632
SC     78  0.205128
GO     76  0.171053
PB     74  0.094595
MS     75  0.080000
CE     73  0.068493
ES   

## Validação temporal complementar
Os meses de fevereiro a julho calibram limiares, e agosto-setembro são os meses não utilizados na estimação. Esta análise complementa, sem substituir, a validação cruzada do roteiro.

In [5]:
ttr, tte, _ = temporal_split(p)

tmodel = make_pipeline(
    OneHotEncoder(handle_unknown='ignore'),
    DecisionTreeClassifier(
        max_depth=4,
        criterion='entropy',
        random_state=42
    )
)

tmodel.fit(class_features(ttr), ttr.alto)

temporal_acc = tmodel.score(
    class_features(tte),
    tte.alto
)

baseline_temporal = accuracy_score(
    tte.alto,
    np.full(len(tte), int(ttr.alto.mean() >= 0.5))
)

print("Acurácia temporal ago-set:", temporal_acc)
print("Baseline temporal:", baseline_temporal)

metrics = {
    'n_painel': len(p),
    'train_n': len(tr),
    'test_n': len(te),
    'id3_train': tr_acc,
    'id3_test': id3_acc,
    'sk_test': sk_acc,
    'cv5_scores': scores,
    'cv5_mean': float(np.mean(scores)),
    'cv5_baseline': float(np.mean(fold_baselines)),
    'baseline_test': bacc,
    'uf_majority_test': ufacc,
    'temporal_test': temporal_acc,
    'root_attribute': max(
        CATEGORICAS,
        key=lambda f: gain(a, f, 'alto')
    ),
    'root_gain': max(
        gain(a, f, 'alto') for f in CATEGORICAS
    ),
    'levels': [1, 2, 3],
    'uf_class_rates': stats['mean'].to_dict()
}

write_metrics('etapa1.json', metrics)

Acurácia temporal ago-set: 0.8653250773993808
Baseline temporal: 0.7801857585139319


PosixPath('FIA_P3_SegurancaPublica_Equipe5/outputs/etapa1.json')

## Não monotonicidade — exemplo formal e implicações
Se uma folha contém as classes `[1,1,1,0]`, a maioria é 1. Se chegam três exemplos legítimos, com a mesma descrição de entrada e classe 0, a maioria passa a 0. Logo, uma conclusão **indutiva** antiga não é irrevogável. Não se trata de negar uma proposição factual: a hipótese foi alterada pelos dados.

Além disso, a indução de ID3 recalcula entropias e ganhos quando novos registros são inseridos. Pode mudar o atributo da raiz, a estrutura dos ramos e os rótulos majoritários; a escolha de P75 também muda se recalculada para outra janela. Portanto, uma regra `UF=XX -> alto` não tem garantia lógica de persistência. Esse fenômeno distingue o raciocínio estatístico revisável da monotonicidade da consequência lógica clássica, na qual adicionar premissas não retira teoremas anteriormente demonstráveis. O uso de probabilidades por folha e a análise de estabilidade temporal mitigam interpretações categóricas injustificadas. As regras ainda refletem diferenças de escala entre estados e disponibilidade administrativa de registro, não um risco individual.

In [6]:
old=pd.Series([1,1,1,0]);new=pd.concat([old,pd.Series([0,0,0])],ignore_index=True)
print('Antes',old.mode().iloc[0],'depois',new.mode().iloc[0]);assert old.mode().iloc[0]!=new.mode().iloc[0]

Antes 1 depois 0


## Verificação complementar de escala: IBGE 2026
As estimativas populacionais oficiais das 27 UFs permitem calcular uma taxa **mensal e específica para cada indicador** por 100 mil habitantes. É uma análise de sensibilidade: os resultados principais permanecem no alvo originalmente definido, em contagens. O novo percentil também é ajustado exclusivamente no treino.

In [7]:
from sklearn.metrics import accuracy_score
from utils import write_metrics
from pathlib import Path
import json

q_rate = tr.groupby('evento').taxa_por100mil.quantile(0.75)

yrate_tr = (
    tr.taxa_por100mil > tr.evento.map(q_rate)
).astype(int)

yrate_te = (
    te.taxa_por100mil > te.evento.map(q_rate)
).astype(int)

model_rate = ID3(max_depth=4).fit(
    class_features(tr),
    yrate_tr
)

acc_rate = accuracy_score(
    yrate_te,
    model_rate.predict(class_features(te))
)

ufrate = (
    yrate_tr.groupby(tr.uf)
    .mean()
    .ge(0.5)
    .astype(int)
)

rate_uf_base = accuracy_score(
    yrate_te,
    te.uf.map(ufrate)
    .fillna(int(yrate_tr.mean() >= 0.5))
    .astype(int)
)

print("Alvo com taxa por 100 mil:")
print("ID3 teste:", acc_rate)
print("Baseline maioria por UF:", rate_uf_base)

m = json.loads(
    (ROOT / 'outputs' / 'etapa1.json').read_text()
)

m.update({
    'id3_rate_test': acc_rate,
    'uf_rate_baseline_test': rate_uf_base,
    'source_ibge': 'IBGE 2026, POP2026_20261029.pdf'
})

write_metrics('etapa1.json', m)

Alvo com taxa por 100 mil:
ID3 teste: 0.8359073359073359
Baseline maioria por UF: 0.7606177606177607


PosixPath('FIA_P3_SegurancaPublica_Equipe5/outputs/etapa1.json')